In [ ]:
import torch
import numpy as np
import math
import torch.nn as nn
import torch.nn.functional as F
from math import log
torch.set_default_dtype(torch.float64)
device=torch.device('cuda:5')
epsilon = 1e-15
class MaskedLinear(nn.Linear):
    def __init__(self, in_channels, out_channels, n, bias, exclusive):
        """初始化带掩码的线性层
        Args:
            in_channels: 输入通道数
            out_channels: 输出通道数
            n: 序列长度或特征维度数
            exclusive: 掩码是否为"严格下三角"（True时输出不依赖当前维度，仅依赖前面维度）
        """
        # 调用父类构造函数，输入维度为in_channels*n，输出维度为out_channels*n
        super(MaskedLinear, self).__init__(in_channels * n, out_channels * n, bias)
        self.in_channels = in_channels    # 记录输入通道数
        self.out_channels = out_channels  # 记录输出通道数
        self.n = n                        # 记录特征维度数
        self.exclusive = exclusive        # 记录掩码类型
        # 初始化掩码（n×n矩阵），初始值全为1
        self.register_buffer('mask', torch.ones([self.n] * 2))
        if self.exclusive:
            # 严格下三角掩码：输出i只依赖输入j < i（排除当前维度）
            self.mask = 1 - torch.triu(self.mask)  # triu生成上三角矩阵，1-后得到严格下三角
        else:
            # 下三角掩码：输出i可依赖输入j ≤ i（包括当前维度）
            self.mask = torch.tril(self.mask)      # tril生成下三角矩阵
            
        # 扩展掩码以匹配输入/输出通道：
        # 沿列方向拼接in_channels次（适配输入通道），沿行方向拼接out_channels次（适配输出通道）
        self.mask = torch.cat([self.mask] * in_channels, dim=1)
        self.mask = torch.cat([self.mask] * out_channels, dim=0)

        # 将掩码应用到权重矩阵（过滤掉不需要的连接）
        self.weight.data *= self.mask

        # 修正Xavier初始化：根据有效连接数（掩码中1的数量）调整权重尺度
        self.weight.data *= torch.sqrt(self.mask.numel() / self.mask.sum())

    def forward(self, x):
        """前向传播：应用掩码后的线性变换
        Args:
            x: 输入张量，形状为[batch_size, in_channels*n]
        Returns:
            线性变换结果，形状为[batch_size, out_channels*n]
        """
        # 每次前向传播都显式应用掩码（确保权重未被意外修改）
        return nn.functional.linear(x, self.mask * self.weight, self.bias)


class MADE(nn.Module):
    """MADE（Masked Autoencoder for Distribution Estimation）模型
    
    用于概率密度估计的自回归模型，通过掩码神经网络实现对高维数据分布的建模，
    支持通过自回归采样生成新数据
    """
    def __init__(self, **kwargs):
        """初始化MADE模型
        
        Args:
            kwargs包含以下参数：
                n: 输入数据的维度（特征数）
                net_depth: 网络深度（层数）
                net_width: 隐藏层宽度（通道数）
                epsilon: 计算对数概率时的微小值（避免log(0)）
                device: 模型运行的设备（CPU/GPU）
        """
        super(MADE, self).__init__()
        self.n = kwargs['n']                  # 输入维度
        self.net_depth = kwargs['net_depth']  # 网络深度
        self.net_width = kwargs['net_width']  # 隐藏层宽度
        self.epsilon = kwargs['epsilon']      # 数值稳定参数
        self.device = kwargs['device']        # 运行设备
        
        # 构建网络层
        layers = []
        # 第一层：输入层到隐藏层（或输出层，若深度为1），使用严格下三角掩码
        layers.append(
            MaskedLinear(
                1,  # 输入通道数为1
                # 若深度为1，直接输出到1通道；否则输出到net_width通道
                1 if self.net_depth == 1 else self.net_width,
                self.n,
                bias=False,
                exclusive=True  # 第一层使用严格下三角掩码（排除当前维度）
            )
        )
        # 中间层：根据深度添加隐藏层
        for count in range(self.net_depth - 2):
            # 使用简单块
            layers.append(self._build_simple_block(self.net_width, self.net_width))

        # 最后一层：隐藏层到输出层（若深度≥2）
        if self.net_depth >= 2:
            layers.append(self._build_simple_block(self.net_width, 1))

        # 输出层后接Sigmoid，将结果映射到(0,1)区间（表示概率）
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)  # 组合所有层为序列模型

    def _build_simple_block(self, in_channels, out_channels,use_dropout=True):
        """构建简单网络块（激活函数 + 掩码线性层）
        Args:
            in_channels: 输入通道数
            out_channels: 输出通道数
        Returns:
            由PReLU和MaskedLinear组成的序列模块
        """
        layers = []
        # PReLU激活函数（带参数的ReLU，更灵活）
        layers.append(nn.PReLU(in_channels * self.n, init=0.5))
        # 掩码线性层（非严格下三角，允许依赖当前维度）
        layers.append(
            MaskedLinear(
                in_channels,
                out_channels,
                self.n,
                bias=False,
                exclusive=False
            )
        )
        return nn.Sequential(*layers)

    def forward(self, x):
        """前向传播：预测每个维度的条件概率
        Args:
            x: 输入张量，形状为[batch_size, n]
        Returns:
            x_hat: 预测的条件概率，形状为[batch_size, n]，每个元素∈(0,1)
        """
        x = x.view(x.shape[0], -1)  # 展平为[batch_size, n]
        x_hat = self.net(x)         # 通过网络得到原始预测
        return x_hat

    def sample(self, batch_size):
        """自回归采样：生成符合模型分布的样本
        按order顺序依次采样每个维度，每个维度的采样依赖于已采样的前面维度
        Args:
            batch_size: 生成样本的批次大小
        Returns:
            sample: 生成的样本，形状为[batch_size, n]，元素为±1（分别表示两种状态）
            x_hat: 采样过程中预测的概率，形状为[batch_size, n]
        """
        # 初始化样本为全0（后续会被替换为±1）
        sample = torch.zeros([batch_size, self.n],device=self.device)
        # 按顺序逐个维度采样
        for i in range(self.n):
            # 预测当前条件下的概率
            x_hat = self.forward(sample)
            # 对第i个维度进行伯努利采样（0/1），再转换为±1
            sample[:, i] = torch.bernoulli(x_hat[:, i])* 2 - 1
        return sample, x_hat

    def log_p(self, sample):
        """计算单个样本的对数概率
        Args:
            sample: 样本张量，形状为[batch_size, n]，元素为±1
        Returns:
            每个样本的对数概率，形状为[batch_size]
        """
        # 将样本从±1转换为0/1掩码（+1→1，-1→0）
        x_hat = self.forward(sample)
        mask = (sample + 1) / 2
        # 计算对数似然：mask部分用log(x_hat)，其余用log(1-x_hat)
        log_prob = (torch.log(x_hat + self.epsilon) * mask +
                    torch.log(1 - x_hat + self.epsilon) * (1 - mask))
        # 对所有维度求和，得到每个样本的总对数概率
        log_prob = log_prob.view(log_prob.shape[0], -1).sum(dim=1)
        return log_prob
def sample_sk(spins,J,num_samples, beta, burn_in,start_beta,beta_anneal=0.99, device=device):
    """
    使用退火 Metropolis 算法从 SK 模型生成自旋构型样本。

    Args:
        J (torch.Tensor): 形状为 (N, N) 的相互作用矩阵，对称且对角元为零。
        num_samples (int): 所需样本数，同时也是并行链的数目。
        beta (float): 逆温度。
        burn_in (int): 每条链的热化步数。
        device (torch.device, optional): 计算设备。若为 None 则自动从 J 获取。

    Returns:
        torch.Tensor: 形状为 (num_samples, N) 的自旋构型，每个元素为 +1 或 -1。
    """
    if device is None:
        device = J.device
    
    N = J.shape[0]

    # ---------- 初始化所有链的自旋构型 (num_samples, N) ----------
    # 随机生成 ±1
    

    # ---------- 计算每条链的初始局域场 ----------
    # 局域场 h_{m,i} = Σ_j J_{i,j} * s_{m,j}
    # 利用矩阵乘法: spins (M,N) @ J.t() (N,N) -> (M,N)
    local_fields = torch.matmul(spins, J.t())  # (num_samples, N)

    # ---------- 热化阶段：并行更新所有链 ----------
    for step in range(burn_in):
        current_beta = (beta-start_beta) * (1 - beta_anneal**step)+start_beta
        # 每条链独立选择一个随机自旋索引
        idx = torch.randint(0, N, (num_samples,), device=device)  # (num_samples,)

        # 获取当前选中的自旋值及其局域场
        spin_i = spins.gather(1, idx.unsqueeze(1)).squeeze(1)   # (num_samples,)
        h_i    = local_fields.gather(1, idx.unsqueeze(1)).squeeze(1)  # (num_samples,)

        # 计算翻转该自旋的能量变化 ΔE = 2 * s_i * h_i
        delta_E = 2 * spin_i * h_i  # (num_samples,)

        # Metropolis 接受准则：若 ΔE ≤ 0 或以概率 exp(-βΔE) 接受
        rand_vals = torch.rand(num_samples, device=device)
        accept = (delta_E <= 0) | (rand_vals < torch.exp(-current_beta * delta_E))  # (num_samples,) 布尔型

        # ---------- 更新接受翻转的链 ----------
        # 新自旋值：接受则取反，否则保持不变
        new_spin_i = torch.where(accept, -spin_i, spin_i)  # (num_samples,)

        # 将新自旋值写回 spins 矩阵
        spins.scatter_(1, idx.unsqueeze(1), new_spin_i.unsqueeze(1))

        # ---------- 更新所有链的局域场 ----------
        # 对于接受翻转的链，自旋变化量 Δs_i = -2 * s_i (旧值)
        # 对其他自旋 j 的局域场影响：Δh_j = J_{j,i} * Δs_i
        delta_spin = torch.where(accept, -2 * spin_i, torch.zeros_like(spin_i))  # (num_samples,)

        # 计算局域场增量：delta_local (num_samples, N)
        # J[:, idx] 形状 (N, num_samples) → 转置后 (num_samples, N)
        # 逐链乘以 delta_spin 的对应元素
        delta_local = delta_spin.unsqueeze(1) * J[:, idx].t()  # (num_samples, N)
        local_fields += delta_local
        
    # 热化结束，返回所有链的最终构型作为样本
    return spins

In [ ]:
#模拟采样
import secrets
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch.nn.functional as F
import numpy as np

T=1.20
n=100
num_T=11
rate=0.05
num_samples=20000
start_burn=100000
burn_in=5000
x=[]
y=[]

J=J.to(device=device)
#初始化 spins
spins = torch.randint(0, 2, (int(num_samples/2), n), device=device).double()
spins = 2 * spins - 1  # 将 {0,1} 映射到 {-1,1}
spins=torch.cat([spins, -spins], dim=0)
spins=sample_sk(spins,J, num_samples, 1/T, start_burn,start_beta=0,beta_anneal=0.99,device=device)

for i in range(num_T):
    spins = sample_sk(spins,J, num_samples,1/T, burn_in,start_beta=1/(T+rate),beta_anneal=0.99,device=device)
    s=spins
    E=-0.5 * torch.sum(s * (s@ J), dim=1)
    Cv=((E**2).mean()-(E.mean())**2)/(T**2)
    y.append(Cv)
    x.append(T)
    print(round(T,2))
    T-=rate
y=(torch.stack(y)).cpu()
plt.plot(x,y,marker='o')
plt.axvline(0.95,color="green", linestyle='--')
plt.xlabel('T')
plt.ylabel('Cv')
plt.show()

In [ ]:
%matplotlib notebook
#Generating network
import torch
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

n=100
kwargs={'n':n,
        'net_depth':2,
        'net_width':5,
        'epsilon':epsilon,
        'device':device
    
}
T=0.70
epochs = 8000
lr = 0.0001
batch_size=10000
seed=69
plt.ion()
fig, ax = plt.subplots()
x_data, y_data=[],[]
line, = ax.plot([], [], 'b-')
ax.set_xlabel('Epochs')
ax.set_ylabel('loss')

#load samples（spl）
spl=torch.load(f"samples")[f"T:{T:.2f}"].to(device=device)
# 初始化模型
torch.manual_seed(seed)
train_time = 0
net =  MADE(**kwargs)
net=net.to(device=device)
# # # 定义优化器
params = net.parameters()
optimizer = torch.optim.Adam(params, lr=lr, betas=(0.9, 0.999))

#训练模型
for epoch in range(epochs):
    #梯度清零
    net.train()
    trainstart_time = time.time()
    optimizer.zero_grad()
    with torch.no_grad():
        indices = torch.randperm(spl.size(0))[:batch_size]
        tensors=spl[indices]
    assert not tensors.requires_grad
    log_q=net.log_p(tensors)
    loss_reinforce= (-log_q).mean()
    loss_reinforce.backward()
    optimizer.step()  
    train_time +=  time.time()-trainstart_time    
    x_data.append(epoch)
    y_data.append(loss_reinforce.item())
    line.set_xdata(x_data)
    line.set_ydata(y_data)
    ax.relim()
    ax.autoscale_view()                   
    fig.canvas.draw()
    fig.canvas.flush_events()
plt.ioff()
plt.show()

In [ ]:
#calculate logp.std
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.interpolate import UnivariateSpline
T=0.70
a=0.05
n=100
y=[]
x=[]
num_T=11
for i in range (num_T):
    logp=torch.load(f"{T:.2f}_logp").to(device=device)
    y.append(logp.std())
    x.append(T)
    T+=a
y=(torch.stack(y)).cpu()
plt.plot(x,y3,marker='o',label="log_p.std")
plt.legend()
plt.show()
